# 台股 ML 預測 — FT-Transformer 訓練

**模型**: FT-Transformer（PyTorch 手刻）
**任務**: 二元分類 × 2（做多模型 / 放空模型）
**不平衡處理**: 不使用 SMOTE，僅靠 `BCEWithLogitsLoss(pos_weight)` 補償（訓練集）
**評估**: Top-K Precision（per rev_date 橫斷面）
**時間切分**:
```
Train : ≤ 2022（所有可用年份）
Test  : 2023, 2024（無驗證集）
```

In [ ]:
import sys

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/機器學習/ds_ml_stock/'
print(f'BASE: {BASE}')

In [ ]:
X        = pd.read_parquet(BASE + 'X_features.parquet')
y_top    = pd.read_parquet(BASE + 'y_top.parquet').squeeze()
y_bottom = pd.read_parquet(BASE + 'y_bottom.parquet').squeeze()
y_return = pd.read_parquet(BASE + 'y_return.parquet').squeeze()

bins_cols = [c for c in X.columns if c.endswith('_bins')]
cont_cols = [c for c in X.columns if not c.endswith('_bins')]

cont_indices = [X.columns.get_loc(c) for c in cont_cols]
cat_indices  = [X.columns.get_loc(c) for c in bins_cols]

n_cont            = len(cont_cols)
cat_cardinalities = [6] * len(bins_cols)

print(f'連續特徵: {n_cont}　類別特徵: {len(bins_cols)}')
print(f'總樣本數: {len(X):,}　日期數: {X.index.get_level_values("datetime").nunique()}')
available_years = sorted(X.index.get_level_values('datetime').year.unique())
print(f'資料年份: {available_years}')

## 時間切分

In [ ]:
years = X.index.get_level_values('datetime').year

train_mask = years <= 2022
test_mask  = years.isin([2023, 2024])

X_train, X_test = X[train_mask], X[test_mask]
y_top_train, y_top_test = y_top[train_mask], y_top[test_mask]
y_bot_train, y_bot_test = y_bottom[train_mask], y_bottom[test_mask]
y_ret_train, y_ret_test = y_return[train_mask], y_return[test_mask]

for name, y in [('Train', y_top_train), ('Test', y_top_test)]:
    print(f'{name:5s}: {len(y):>7,} 樣本  正類={y.sum():>5,.0f} ({y.mean():.2%})')

## 訓練資料準備

不使用 SMOTE-Tomek，直接以原始不平衡資料訓練。
類別不平衡由 `BCEWithLogitsLoss(pos_weight)` 動態補償。

In [ ]:
X_top_res = X_train.values.astype(float)
y_top_res = y_top_train.values.astype(float)

X_bot_res = X_train.values.astype(float)
y_bot_res = y_bot_train.values.astype(float)

print(f'Top  正類={y_top_res.sum():,.0f}  負類={(1-y_top_res).sum():,.0f}  比例={y_top_res.mean():.3%}')
print(f'Bot  正類={y_bot_res.sum():,.0f}  負類={(1-y_bot_res).sum():,.0f}  比例={y_bot_res.mean():.3%}')

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X_np: np.ndarray, y_np: np.ndarray,
                 cont_indices: list, cat_indices: list):
        self.X_cont = torch.FloatTensor(X_np[:, cont_indices])
        self.X_cat  = torch.LongTensor(X_np[:, cat_indices].astype(int))
        self.y      = torch.FloatTensor(y_np)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_cont[idx], self.X_cat[idx], self.y[idx]


def make_loader(X_np, y_np, cont_indices, cat_indices, batch_size, shuffle=True):
    ds = TabularDataset(X_np, y_np, cont_indices, cat_indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=True)

## FT-Transformer（PyTorch 手刻）

**架構**
```
x_cont, x_cat
  ↓ FeatureTokenizer
    連續: x_j × W_j + b_j  →  (B, n_cont, d)
    類別: Embedding[i](x_cat[:,i])  →  (B, n_cat, d)
    concat  →  (B, n_feat, d)
  ↓ prepend [CLS] token  →  (B, 1+n_feat, d)
  ↓ TransformerEncoder（Pre-LN, GELU）× n_layers
  ↓ 取 CLS output[:, 0, :]  →  (B, d)
  ↓ LayerNorm → Linear → GELU → Dropout → Linear
  →  logit (B,)
```

In [ ]:
class FeatureTokenizer(nn.Module):
    def __init__(self, n_cont: int, cat_cardinalities: list, d_token: int):
        super().__init__()
        # 每個連續特徵有自己的 weight 向量和 bias 向量
        self.cont_W = nn.Parameter(torch.empty(n_cont, d_token))
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d_token))
        nn.init.kaiming_uniform_(self.cont_W, a=math.sqrt(5))

        # 每個類別特徵有自己的 Embedding table
        self.cat_emb = nn.ModuleList([
            nn.Embedding(card, d_token)
            for card in cat_cardinalities
        ])

    def forward(self, x_cont: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        # x_cont: (B, n_cont)  float
        # x_cat:  (B, n_cat)   long
        t_cont = x_cont.unsqueeze(-1) * self.cont_W + self.cont_b   # (B, n_cont, d)
        t_cat  = torch.stack(
            [self.cat_emb[i](x_cat[:, i]) for i in range(x_cat.shape[1])],
            dim=1
        )                                                             # (B, n_cat, d)
        return torch.cat([t_cont, t_cat], dim=1)                     # (B, n_feat, d)


class FTTransformer(nn.Module):
    def __init__(
        self,
        n_cont: int,
        cat_cardinalities: list,
        d_token: int = 192,
        n_heads: int = 8,
        n_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_token % n_heads == 0, 'd_token 必須能被 n_heads 整除'

        self.tokenizer = FeatureTokenizer(n_cont, cat_cardinalities, d_token)

        # 可學習的 [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Pre-LN Transformer（比 Post-LN 訓練更穩定）
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # 輸出頭（接在 CLS token 上）
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token // 2, 1),
        )

    def forward(self, x_cont: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        tokens = self.tokenizer(x_cont, x_cat)                        # (B, n_feat, d)
        cls    = self.cls_token.expand(tokens.size(0), -1, -1)        # (B, 1, d)
        tokens = torch.cat([cls, tokens], dim=1)                      # (B, 1+n_feat, d)
        out    = self.transformer(tokens)                              # (B, 1+n_feat, d)
        return self.head(out[:, 0]).squeeze(-1)                        # (B,)  logit

## 評估指標 — Top-K Precision

每個 rev_date 獨立計算：預測分數最高的 K 支股票中，真正屬於前（後）1% 的比例。  
K 預設等於當日股票數的 1%（與 label 定義對齊），也可指定固定 K。

In [ ]:
def top_k_precision(
    X_df: pd.DataFrame,
    y_series: pd.Series,
    model: nn.Module,
    cont_indices: list,
    cat_indices: list,
    k_pct: float = 0.01,
    batch_size: int = 1024,
    device=DEVICE,
) -> float:
    """
    Per rev_date 計算 Top-K Precision，回傳所有日期的平均值。

    k_pct=0.01 → 每天取預測分數最高的 1% 股票，計算其中有幾支是真正的正類。
    """
    model.eval()
    X_np = X_df.values.astype(float)

    # 批次推論（避免 OOM）
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xc = torch.FloatTensor(X_np[i:i+batch_size, cont_indices]).to(device)
            xk = torch.LongTensor(X_np[i:i+batch_size, cat_indices].astype(int)).to(device)
            logits = model(xc, xk)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)

    probs_all = np.concatenate(all_probs)

    # 組合成 DataFrame，按 datetime 分組
    result_df = pd.DataFrame({
        'prob' : probs_all,
        'label': y_series.values,
    }, index=X_df.index)

    precisions = []
    for dt, grp in result_df.groupby(level='datetime'):
        k = max(1, int(np.ceil(len(grp) * k_pct)))
        top_k = grp.nlargest(k, 'prob')
        precisions.append(top_k['label'].mean())

    return float(np.mean(precisions))

## 訓練與驗證函數

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x_cont, x_cat, y in loader:
        x_cont, x_cat, y = x_cont.to(device), x_cat.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x_cont, x_cat)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


def compute_loss(model, X_df, y_series, cont_indices, cat_indices,
                 criterion, device, batch_size=1024):
    """在指定資料集上計算平均 BCE loss（不更新梯度）。"""
    model.eval()
    X_np = X_df.values.astype(float)
    y_np = y_series.values.astype(float)
    total_loss = 0.0
    with torch.no_grad():
        for i in range(0, len(y_np), batch_size):
            xc = torch.FloatTensor(X_np[i:i+batch_size, cont_indices]).to(device)
            xk = torch.LongTensor(X_np[i:i+batch_size, cat_indices].astype(int)).to(device)
            yb = torch.FloatTensor(y_np[i:i+batch_size]).to(device)
            total_loss += criterion(model(xc, xk), yb).item() * len(yb)
    return total_loss / len(y_np)


def train_model(
    model_name: str,
    X_train_res: np.ndarray,
    y_train_res: np.ndarray,
    X_test_df: pd.DataFrame,
    y_test: pd.Series,
    cont_indices: list,
    cat_indices: list,
    cat_cardinalities: list,
    config: dict,
    device=DEVICE,
) -> nn.Module:

    # pos_weight 從 resampled 資料動態計算
    n_pos = y_train_res.sum()
    n_neg = len(y_train_res) - n_pos
    pos_w = torch.tensor(n_neg / n_pos, dtype=torch.float32).to(device)
    print(f'[{model_name}] pos_weight = {pos_w.item():.3f}  '
          f'(neg={n_neg:,} / pos={n_pos:,})')

    loader = make_loader(
        X_train_res, y_train_res,
        cont_indices, cat_indices,
        batch_size=config['batch_size'],
    )

    model = FTTransformer(
        n_cont=len(cont_indices),
        cat_cardinalities=cat_cardinalities,
        d_token=config['d_token'],
        n_heads=config['n_heads'],
        n_layers=config['n_layers'],
        dropout=config['dropout'],
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['epochs']
    )

    train_losses = []
    test_losses  = []

    for epoch in range(1, config['epochs'] + 1):
        train_loss = train_one_epoch(model, loader, optimizer, criterion, device)
        test_loss  = compute_loss(
            model, X_test_df, y_test,
            cont_indices, cat_indices, criterion, device,
        )
        scheduler.step()

        train_losses.append(train_loss)
        test_losses.append(test_loss)

        if epoch % 5 == 0:
            print(f'  Epoch {epoch:3d} | train={train_loss:.4f} | test={test_loss:.4f}')

    # ── Loss 曲線圖 ──────────────────────────────────────
    epochs_x = range(1, len(train_losses) + 1)
    plt.figure(figsize=(9, 4))
    plt.plot(epochs_x, train_losses, label='Train Loss', linewidth=1.5)
    plt.plot(epochs_x, test_losses,  label='Test Loss',  linewidth=1.5)
    plt.xlabel('Epoch')
    plt.ylabel('BCE Loss (weighted)')
    plt.title(f'{model_name} Model — Train / Test Loss Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return model

## 訓練設定

In [ ]:
CONFIG = {
    # 模型架構
    'd_token'      : 192,
    'n_heads'      : 8,
    'n_layers'     : 3,
    'dropout'      : 0.1,
    # 訓練
    'lr'           : 1e-4,
    'weight_decay' : 1e-5,
    'batch_size'   : 256,
    'epochs'       : 100,
    'patience'     : 10,
    # 評估
    'k_pct'        : 0.01,   # Top-K Precision：每日前 1%
}

## 訓練 Top Model（做多訊號）

In [ ]:
print('=' * 55)
print('  TOP MODEL — 預測報酬前 1%（做多）')
print('=' * 55)

model_top = train_model(
    model_name        = 'Top',
    X_train_res       = X_top_res,
    y_train_res       = y_top_res,
    X_test_df         = X_test,
    y_test            = y_top_test,
    cont_indices      = cont_indices,
    cat_indices       = cat_indices,
    cat_cardinalities = cat_cardinalities,
    config            = CONFIG,
    device            = DEVICE,
)

## 訓練 Bottom Model（放空訊號）

In [ ]:
print('=' * 55)
print('  BOTTOM MODEL — 預測報酬後 1%（放空）')
print('=' * 55)

model_bot = train_model(
    model_name        = 'Bottom',
    X_train_res       = X_bot_res,
    y_train_res       = y_bot_res,
    X_test_df         = X_test,
    y_test            = y_bot_test,
    cont_indices      = cont_indices,
    cat_indices       = cat_indices,
    cat_cardinalities = cat_cardinalities,
    config            = CONFIG,
    device            = DEVICE,
)

## Test 最終評估

In [ ]:
test_top_prec = top_k_precision(
    X_test, y_top_test, model_top,
    cont_indices, cat_indices, k_pct=CONFIG['k_pct'],
)
test_bot_prec = top_k_precision(
    X_test, y_bot_test, model_bot,
    cont_indices, cat_indices, k_pct=CONFIG['k_pct'],
)

# 隨機基準：若隨機選 1%，期望 precision = 1%
baseline = CONFIG['k_pct']

print('=' * 45)
print('  TEST SET 最終結果')
print('=' * 45)
print(f'  Top    model | Top-K Prec = {test_top_prec:.4f}  (baseline={baseline:.4f})')
print(f'  Bottom model | Top-K Prec = {test_bot_prec:.4f}  (baseline={baseline:.4f})')
print(f'  Top    lift = {test_top_prec/baseline:.2f}x')
print(f'  Bottom lift = {test_bot_prec/baseline:.2f}x')

In [ ]:
def save_model(model, path, cont_cols, bins_cols, config):
    torch.save({
        'state_dict'       : model.state_dict(),
        'cont_cols'        : cont_cols,
        'bins_cols'        : bins_cols,
        'cat_cardinalities': cat_cardinalities,
        'config'           : config,
    }, path)
    print(f'✅ 儲存: {path}')

save_model(model_top, BASE + 'model_top.pt',    cont_cols, bins_cols, CONFIG)
save_model(model_bot, BASE + 'model_bottom.pt', cont_cols, bins_cols, CONFIG)